In [1]:
import dask_geopandas as dgpd
import geopandas as gpd
import rasterio
from rasterio.sample import sample_gen
from pathlib import Path
import numpy as np
import pandas as pd
import dask

In [2]:
# 📂 Шлях до геоморфонів
geomorphon_folder = Path("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/geomorphons")
geomorphon_paths = sorted(geomorphon_folder.glob("*_geomorphons.tif"))

In [3]:
geomorphon_paths

[PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/geomorphons/alos_dem_utm32635_geomorphons.tif'),
 PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/geomorphons/aster_dem_utm32635_geomorphons.tif'),
 PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/geomorphons/copernicus_dеm_utm32635_geomorphons.tif'),
 PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/geomorphons/fab_dem_utm32635_geomorphons.tif'),
 PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/geomorphons/nasa_dem_utm32635_geomorphons.tif'),
 PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/geomorphons/srtm_dem_utm32635_geomorphons.tif'),
 PosixPath('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/DATA_UTM/dem/geomorphons/tan_dem_utm32635_geomorphons.tif')]

In [4]:
# 📖 ICESat дані
ice_gdf = gpd.read_parquet("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_dems_delta_32635.parquet")


In [5]:
coords = [(geom.x, geom.y) for geom in ice_gdf.geometry]


In [6]:
landform_names = {
    1: "Flat",
    2: "Peak",
    3: "Ridge",
    4: "Shoulder",
    5: "Spur",
    6: "Slope",
    7: "Hollow",
    8: "Footslope",
    9: "Valley",
    10: "Pit"
}

In [8]:
# 🔁 Проходимо по всіх геоморфон-DEM
for path in geomorphon_paths:
    filename = path.stem  # приклад: tan_dem_utm32635_geomorphons
    parts = filename.split("_")
    dem_name = "_".join(parts[:2])  # tan_dem
    col_class = f"{dem_name}_geomorphon"
    col_name = f"{dem_name}_landform"

    # Читання та витяг значень
    with rasterio.open(path) as src:
        if not src.crs == ice_gdf.crs:
            raise ValueError(f"❌ CRS не збігається: {col_class}")
        sampled = list(src.sample(coords))

    # Значення форм
    geomorph_classes = [val[0] if val and val[0] > 0 else np.nan for val in sampled]
    geomorph_names = [
        landform_names.get(int(val), None) if not np.isnan(val) else None for val in geomorph_classes
    ]

    # Додавання до GeoDataFrame
    ice_gdf[col_class] = geomorph_classes
    ice_gdf[col_name] = geomorph_names

    print(f"✅ Додано: {col_class}, {col_name}")

# 💾 Збереження
out_path = "/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_geomorphons.parquet"
ice_gdf.to_parquet(out_path)
print(f"📦 Збережено з геоморфонами: {out_path}")


✅ Додано: alos_dem_geomorphon, alos_dem_landform
✅ Додано: aster_dem_geomorphon, aster_dem_landform
✅ Додано: copernicus_dеm_geomorphon, copernicus_dеm_landform
✅ Додано: fab_dem_geomorphon, fab_dem_landform
✅ Додано: nasa_dem_geomorphon, nasa_dem_landform
✅ Додано: srtm_dem_geomorphon, srtm_dem_landform
✅ Додано: tan_dem_geomorphon, tan_dem_landform
📦 Збережено з геоморфонами: /mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_geomorphons.parquet
